# Module 2: When Perfect Training = Disaster

In Module 1 we explored the dataset — 254 countries described by 493 features. Now we ask: **can we just throw all features into a linear regression and predict GDP?**

Spoiler: the model will *memorize* the training data perfectly, yet fail catastrophically on unseen countries. This module builds the intuition for **why**.

In [ ]:
# ── Colab Setup (auto-skipped when running locally) ──────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    GITHUB_RAW_BASE = (
        "https://raw.githubusercontent.com/eth-bmai-fs26/coding-exercises"
        "/week1/cx1_un-games"
    )

    import subprocess, urllib.request

    # 1. Install packages not pre-installed in Colab
    subprocess.run(["pip", "install", "-q", "ipywidgets"], check=True)

    # 2. Download data files into /content (Colab's working directory)
    for fname in ["gdp_spurious_regression_dataset.csv", "codebook.csv"]:
        if not os.path.exists(fname):
            print(f"Downloading {fname}...")
            urllib.request.urlretrieve(f"{GITHUB_RAW_BASE}/{fname}", fname)

    # 3. Write helpers package inline (no GitHub dependency for helper files)
    helpers_dir = "/content/helpers"
    os.makedirs(helpers_dir, exist_ok=True)

    with open(os.path.join(helpers_dir, "__init__.py"), "w") as f:
        f.write(
            "from .data_loader import load_dataset\n"
            "from .styling import (\n"
            "    ROLE_COLORS, DARK_BG, AXES_BG, TEXT_COLOR, ACCENT_BLUE,\n"
            "    apply_dark_theme, takeaway_box, metric_card, role_bar_colors\n"
            ")\n"
        )

    with open(os.path.join(helpers_dir, "data_loader.py"), "w") as f:
        f.write(
            '"""Shared data loading for GDP Spurious Regression notebooks."""\n'
            "import os, pandas as pd\n\n"
            "def _find_file(filename):\n"
            "    cwd_path = os.path.join(os.getcwd(), filename)\n"
            "    if os.path.exists(cwd_path):\n"
            "        return cwd_path\n"
            "    base_dir = os.path.join(os.path.dirname(__file__), '..', '..')\n"
            "    return os.path.join(base_dir, filename)\n\n"
            "def load_dataset():\n"
            "    df = pd.read_csv(_find_file('gdp_spurious_regression_dataset.csv'), index_col=0)\n"
            "    codebook = pd.read_csv(_find_file('codebook.csv'))\n"
            "    y = df['gdp_per_capita_usd']\n"
            "    X = df.drop(columns=['gdp_per_capita_usd'])\n"
            "    role_map = dict(zip(codebook['column_name'], codebook['role']))\n"
            "    return X, y, codebook, role_map\n"
        )

    with open(os.path.join(helpers_dir, "styling.py"), "w") as f:
        f.write(
            '"""Shared styling for GDP Spurious Regression notebooks."""\n'
            "import ipywidgets as widgets\n\n"
            "DARK_BG = '#0a0e27'\n"
            "AXES_BG = '#1a1a2e'\n"
            "TEXT_COLOR = '#e0e7ff'\n"
            "ACCENT_BLUE = '#3b82f6'\n"
            "GREEN = '#22c55e'\n"
            "RED = '#ef4444'\n"
            "GRAY = '#94a3b8'\n"
            "MUTED_TEXT = '#a5b4fc'\n\n"
            "ROLE_COLORS = {\n"
            "    'causal': '#22c55e',\n"
            "    'spurious': '#ef4444',\n"
            "    'incidental': '#94a3b8',\n"
            "    'target': '#3b82f6',\n"
            "}\n\n"
            "def apply_dark_theme(fig, ax):\n"
            "    fig.patch.set_facecolor(DARK_BG)\n"
            "    axes = ax if isinstance(ax, list) else ([ax] if not hasattr(ax, '__iter__') else list(ax.flat))\n"
            "    for a in axes:\n"
            "        a.set_facecolor(AXES_BG)\n"
            "        a.tick_params(colors=MUTED_TEXT, which='both')\n"
            "        a.xaxis.label.set_color(MUTED_TEXT)\n"
            "        a.yaxis.label.set_color(MUTED_TEXT)\n"
            "        a.title.set_color(TEXT_COLOR)\n"
            "        a.grid(True, alpha=0.15, linestyle='--', color='#4a5568')\n"
            "        for spine in a.spines.values():\n"
            "            spine.set_edgecolor('#4a4a6a')\n"
            "            spine.set_linewidth(1.5)\n\n"
            "def takeaway_box(text):\n"
            "    return widgets.HTML(f\"\"\"\n"
            "    <div style='background: linear-gradient(135deg, #1e3a5f, #1a1a2e);\n"
            "                padding: 20px 25px; border-radius: 12px;\n"
            "                border-left: 4px solid {ACCENT_BLUE};\n"
            "                margin: 15px 0; max-width: 900px;'>\n"
            "        <p style='color: {TEXT_COLOR}; font-size: 15px; line-height: 1.6; margin: 0;'>\n"
            "            {text}\n"
            "        </p>\n"
            "    </div>\n"
            "    \"\"\")\n\n"
            "def metric_card(label, value, color=ACCENT_BLUE, width='180px'):\n"
            "    return widgets.HTML(f\"\"\"\n"
            "    <div style='background: #1e293b; padding: 14px 18px; border-radius: 10px;\n"
            "                border: 1px solid #475569; text-align: center;\n"
            "                display: inline-block; width: {width}; margin: 5px;'>\n"
            "        <div style='color: #94a3b8; font-size: 12px; text-transform: uppercase;\n"
            "                    letter-spacing: 1px; margin-bottom: 6px;'>{label}</div>\n"
            "        <div style='color: {color}; font-size: 24px; font-weight: bold;'>{value}</div>\n"
            "    </div>\n"
            "    \"\"\")\n\n"
            "def role_bar_colors(features, role_map):\n"
            "    return [ROLE_COLORS.get(role_map.get(f, 'incidental'), GRAY) for f in features]\n\n"
            "def role_legend_html():\n"
            "    return widgets.HTML(\"\"\"\n"
            "    <div style='display: flex; gap: 20px; margin: 10px 0; align-items: center;'>\n"
            "        <span style='color: #22c55e; font-weight: bold;'>&#9679; Causal</span>\n"
            "        <span style='color: #ef4444; font-weight: bold;'>&#9679; Spurious</span>\n"
            "        <span style='color: #94a3b8; font-weight: bold;'>&#9679; Incidental</span>\n"
            "    </div>\n"
            "    \"\"\")\n\n"
            "def styled_button(description, color=ACCENT_BLUE, width='140px'):\n"
            "    btn = widgets.Button(description=description, layout=widgets.Layout(width=width, height='40px'))\n"
            "    btn.style.button_color = color\n"
            "    btn.style.text_color = 'white'\n"
            "    btn.style.font_weight = 'bold'\n"
            "    return btn\n"
        )

    # 4. Ensure /content is on sys.path so `import helpers` works
    if '/content' not in sys.path:
        sys.path.insert(0, '/content')

print("Setup complete.")

In [ ]:
#@title 🔧 Setup { display-mode: "form" }

import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from helpers import load_dataset, ROLE_COLORS, apply_dark_theme, takeaway_box, metric_card, role_bar_colors
from helpers.styling import role_legend_html, styled_button, DARK_BG, AXES_BG, TEXT_COLOR, GREEN, RED, GRAY, MUTED_TEXT

X, y, codebook, role_map = load_dataset()

---
## 1. OLS on All Features

Ordinary Least Squares (OLS) finds the coefficients that minimize the sum of squared errors on the **training** data. With 493 features and only ~200 training samples, the system is *under-determined* — there are infinitely many perfect solutions.

Use the slider below to pick a test size, then hit **Train OLS** to see what happens.

In [ ]:
#@title 🔧 OLS Demo Widget { display-mode: "form" }

class OLSDemoWidget:
    """Interactive widget demonstrating OLS overfitting on all features."""

    def __init__(self, X, y):
        self.X = X
        self.y = y

        self.test_slider = widgets.IntSlider(
            value=20, min=10, max=50, step=5,
            description='Test size %:',
            style={'description_width': '100px'},
            layout=widgets.Layout(width='400px')
        )
        self.train_btn = styled_button('Train OLS', width='160px')
        self.train_btn.on_click(self._on_train)

        self.output = widgets.Output()

    def _on_train(self, _):
        with self.output:
            clear_output(wait=True)

            test_frac = self.test_slider.value / 100

            X_train, X_test, y_train, y_test = train_test_split(
                self.X, self.y, test_size=test_frac, random_state=42
            )

            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)

            model = LinearRegression()
            model.fit(X_train_s, y_train)

            train_r2 = r2_score(y_train, model.predict(X_train_s))
            test_r2 = r2_score(y_test, model.predict(X_test_s))

            train_color = GREEN if train_r2 > 0.8 else RED
            test_color = GREEN if test_r2 > 0.5 else RED

            cards = widgets.HBox([
                metric_card('Train R²', f'{train_r2:.4f}', color=train_color, width='220px'),
                metric_card('Test R²', f'{test_r2:.4f}', color=test_color, width='220px'),
            ])

            info = widgets.HTML(f"""
            <div style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 10px;'>
                Training samples: {len(X_train)} &nbsp;|&nbsp; Test samples: {len(X_test)}
                &nbsp;|&nbsp; Features: {self.X.shape[1]}
            </div>
            """)

            display(cards, info)

    def create_ui(self):
        title = widgets.HTML(f"""
        <h3 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>OLS with All {self.X.shape[1]} Features</h3>
        <p style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 0;'>
            Adjust the test size and click <b>Train OLS</b> to see the train/test gap.
        </p>
        """)
        controls = widgets.VBox([self.test_slider, self.train_btn])
        return widgets.VBox([title, controls, self.output])

In [ ]:
#@title ▶️ Launch OLS Demo { display-mode: "form" }

ols_demo = OLSDemoWidget(X, y)
display(ols_demo.create_ui())

---
## 2. The Feature Count Experiment

**Key question**: does the problem get worse as we add more features?

When the number of features *p* exceeds the number of training samples *n*, OLS can always find coefficients that achieve a perfect fit on training data — it simply solves an under-determined system. But those coefficients are wildly unstable and produce garbage predictions.

Use the slider to train a single model at a given feature count, or click **Run Sweep** to see the full curve.

In [ ]:
#@title 🔧 Feature Count Widget { display-mode: "form" }

N_FEATURES = X.shape[1]

class FeatureCountWidget:
    """Interactive widget showing how test R² degrades as feature count grows."""

    # Dense sweep: every 10 features up to 300, then every 25 up to N_FEATURES
    SWEEP_COUNTS = (
        list(range(5, 300, 10))
        + list(range(300, N_FEATURES, 25))
        + [N_FEATURES]
    )

    def __init__(self, X, y):
        self.X = X
        self.y = y

        self.feat_slider = widgets.IntSlider(
            value=50, min=5, max=N_FEATURES, step=5,
            description='# Features:',
            style={'description_width': '100px'},
            layout=widgets.Layout(width='400px')
        )
        self.sweep_btn = styled_button('Run Sweep', width='160px')
        self.sweep_btn.on_click(self._on_sweep)

        self.output = widgets.Output()

    def _train_ols(self, n_features):
        """Train OLS with first n_features columns and return train/test R²."""
        X_sub = self.X.iloc[:, :n_features]
        X_train, X_test, y_train, y_test = train_test_split(
            X_sub, self.y, test_size=0.2, random_state=42
        )
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        model = LinearRegression()
        model.fit(X_train_s, y_train)

        train_r2 = r2_score(y_train, model.predict(X_train_s))
        test_r2 = r2_score(y_test, model.predict(X_test_s))
        return train_r2, test_r2

    def _on_sweep(self, _):
        with self.output:
            clear_output(wait=True)

            # Single-value demo from slider
            n_feat = self.feat_slider.value
            single_train_r2, single_test_r2 = self._train_ols(n_feat)

            train_color = GREEN if single_train_r2 > 0.8 else RED
            test_color = GREEN if single_test_r2 > 0.5 else RED

            cards = widgets.HBox([
                metric_card(f'Train R² ({n_feat} feat)', f'{single_train_r2:.4f}',
                            color=train_color, width='220px'),
                metric_card(f'Test R² ({n_feat} feat)', f'{single_test_r2:.4f}',
                            color=test_color, width='220px'),
            ])
            display(cards)

            # Full sweep
            train_scores, test_scores = [], []
            for n in self.SWEEP_COUNTS:
                tr, te = self._train_ols(n)
                train_scores.append(tr)
                test_scores.append(te)

            # Clamp test R² to -1 for readability (some values go to -35)
            test_scores_clamped = [max(te, -1.0) for te in test_scores]
            any_clamped = any(te < -1.0 for te in test_scores)

            n_train = int(len(self.y) * 0.8)

            fig, ax = plt.subplots(figsize=(10, 5))
            apply_dark_theme(fig, ax)

            ax.plot(self.SWEEP_COUNTS, train_scores, '-', color=GREEN,
                    linewidth=2, markersize=3, marker='o', label='Train R²')
            ax.plot(self.SWEEP_COUNTS, test_scores_clamped, '-', color=RED,
                    linewidth=2, markersize=3, marker='s', label='Test R²')

            ax.axvline(x=n_train, color=MUTED_TEXT, linestyle='--', linewidth=1.5, alpha=0.8)
            ax.text(n_train + 5, -0.95, f'p = n ({n_train})',
                    color=MUTED_TEXT, fontsize=11, va='bottom')

            ax.axhline(y=0, color=GRAY, linestyle=':', linewidth=1, alpha=0.5)
            ax.set_ylim(-1.1, 1.1)

            ax.set_xlabel('Number of Features (p)', fontsize=12)
            ax.set_ylabel('R² Score', fontsize=12)
            title = 'The Hockey Stick of Doom: Train vs Test R²'
            if any_clamped:
                title += '  (Test R² clamped at −1)'
            ax.set_title(title, fontsize=14)
            ax.legend(loc='lower left', fontsize=11, facecolor=AXES_BG, edgecolor='#4a4a6a',
                      labelcolor=TEXT_COLOR)

            plt.tight_layout()
            plt.show()

    def create_ui(self):
        title = widgets.HTML(f"""
        <h3 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>Feature Count vs Performance</h3>
        <p style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 0;'>
            Pick a feature count to see its metric cards, then <b>Run Sweep</b> for the full picture.
            Features are added in dataset column order (causal first, then incidental, then spurious).
        </p>
        """)
        controls = widgets.VBox([self.feat_slider, self.sweep_btn])
        return widgets.VBox([title, controls, self.output])

In [ ]:
#@title ▶️ Launch Feature Count Experiment { display-mode: "form" }

feat_widget = FeatureCountWidget(X, y)
display(feat_widget.create_ui())

---
## 3. Actual vs Predicted: A Visual Reality Check

Numbers only tell part of the story. Let us look at the **actual vs predicted** scatter plots side by side for the training and test sets. If the model truly learned the relationship, points should hug the diagonal (y = x) in **both** plots.

In [ ]:
#@title 🔧 Actual vs Predicted Widget { display-mode: "form" }

class ActualVsPredictedWidget:
    """Side-by-side scatter plots of actual vs predicted for train and test."""

    def __init__(self, X, y):
        self.X = X
        self.y = y

        self.show_btn = styled_button('Show Predictions', width='180px')
        self.show_btn.on_click(self._on_show)

        self.output = widgets.Output()

    def _on_show(self, _):
        with self.output:
            clear_output(wait=True)

            X_train, X_test, y_train, y_test = train_test_split(
                self.X, self.y, test_size=0.2, random_state=42
            )

            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)

            model = LinearRegression()
            model.fit(X_train_s, y_train)

            y_train_pred = model.predict(X_train_s)
            y_test_pred = model.predict(X_test_s)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
            apply_dark_theme(fig, [ax1, ax2])

            # Shared axis limits
            all_vals = np.concatenate([y_train.values, y_test.values,
                                       y_train_pred, y_test_pred])
            vmin, vmax = np.min(all_vals), np.max(all_vals)
            pad = (vmax - vmin) * 0.05

            # Training plot
            ax1.scatter(y_train, y_train_pred, c=GREEN, alpha=0.6, s=30, edgecolors='none')
            ax1.plot([vmin - pad, vmax + pad], [vmin - pad, vmax + pad],
                     '--', color='white', linewidth=1.5, alpha=0.7)
            ax1.set_xlim(vmin - pad, vmax + pad)
            ax1.set_ylim(vmin - pad, vmax + pad)
            ax1.set_xlabel('Actual GDP per capita (USD)', fontsize=11)
            ax1.set_ylabel('Predicted GDP per capita (USD)', fontsize=11)
            ax1.set_title('Training Set (Perfect Fit)', fontsize=13)

            # Test plot
            ax2.scatter(y_test, y_test_pred, c=RED, alpha=0.6, s=30, edgecolors='none')
            ax2.plot([vmin - pad, vmax + pad], [vmin - pad, vmax + pad],
                     '--', color='white', linewidth=1.5, alpha=0.7)
            ax2.set_xlim(vmin - pad, vmax + pad)
            ax2.set_xlabel('Actual GDP per capita (USD)', fontsize=11)
            ax2.set_ylabel('Predicted GDP per capita (USD)', fontsize=11)
            ax2.set_title('Test Set (Wild Predictions)', fontsize=13)

            plt.tight_layout()
            plt.show()

            # Show R² cards too
            train_r2 = r2_score(y_train, y_train_pred)
            test_r2 = r2_score(y_test, y_test_pred)
            cards = widgets.HBox([
                metric_card('Train R²', f'{train_r2:.4f}', color=GREEN, width='220px'),
                metric_card('Test R²', f'{test_r2:.4f}', color=RED, width='220px'),
            ])
            display(cards)

    def create_ui(self):
        n_feat = self.X.shape[1]
        title = widgets.HTML(f"""
        <h3 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>Actual vs Predicted: Train &amp; Test</h3>
        <p style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 0;'>
            Click <b>Show Predictions</b> to see scatter plots using all {n_feat} features.
        </p>
        """)
        return widgets.VBox([title, self.show_btn, self.output])

In [ ]:
#@title ▶️ Launch Actual vs Predicted { display-mode: "form" }

avp_widget = ActualVsPredictedWidget(X, y)
display(avp_widget.create_ui())

---
## 4. What Does OLS Think Matters?

Even though OLS overfits terribly, we can still look at which features it assigns the
**largest coefficients** (after standardization). Are they causal, spurious, or incidental?
This foreshadows the deeper analysis in Modules 3 and 4.

In [ ]:
#@title 🔧 OLS Top Features Widget { display-mode: "form" }

class OLSTopFeaturesWidget:
    """Show the top OLS coefficients color-coded by role (causal/spurious/incidental)."""

    def __init__(self, X, y, codebook, role_map):
        self.X = X
        self.y = y
        self.codebook = codebook
        self.role_map = role_map

        self.n_slider = widgets.IntSlider(
            value=30, min=10, max=60, step=5,
            description='Top N:',
            style={'description_width': '80px'},
            layout=widgets.Layout(width='300px')
        )
        self.show_btn = styled_button('Show Top Features', width='180px')
        self.show_btn.on_click(self._on_show)
        self.output = widgets.Output()

    def _on_show(self, _):
        top_n = self.n_slider.value
        with self.output:
            clear_output(wait=True)

            X_train, X_test, y_train, y_test = train_test_split(
                self.X, self.y, test_size=0.2, random_state=42
            )
            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)

            model = LinearRegression()
            model.fit(X_train_s, y_train)

            # Build coefficient dataframe
            coef_df = pd.DataFrame({
                'feature': self.X.columns,
                'coefficient': model.coef_,
                'abs_coef': np.abs(model.coef_),
            }).sort_values('abs_coef', ascending=True).tail(top_n)

            roles = [self.role_map.get(f, 'incidental') for f in coef_df['feature']]
            colors = [ROLE_COLORS.get(r, GRAY) for r in roles]

            # Count by role in top N
            from collections import Counter
            role_counts = Counter(roles)

            fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.28)))
            apply_dark_theme(fig, ax)

            bars = ax.barh(range(len(coef_df)), coef_df['abs_coef'].values,
                           color=colors, alpha=0.85, edgecolor='white', linewidth=0.3)

            ax.set_yticks(range(len(coef_df)))
            ax.set_yticklabels(coef_df['feature'].values, fontsize=9)
            ax.set_xlabel('|Coefficient| (standardized)', fontsize=12)
            ax.set_title(f'Top {top_n} OLS Coefficients by Magnitude', fontsize=14, fontweight='bold')

            # Add role labels on bars
            for i, (_, row) in enumerate(coef_df.iterrows()):
                role = self.role_map.get(row['feature'], 'incidental')
                label = role[0].upper()  # C, S, or I
                ax.text(row['abs_coef'] + ax.get_xlim()[1] * 0.01, i,
                        f' {label}', va='center', fontsize=8,
                        color=ROLE_COLORS.get(role, GRAY), fontweight='bold')

            plt.tight_layout()
            plt.show()

            # Role legend + summary
            display(role_legend_html())

            n_causal = role_counts.get('causal', 0)
            n_spurious = role_counts.get('spurious', 0)
            n_incidental = role_counts.get('incidental', 0)

            summary = widgets.HTML(f"""
            <div style='background: #1e293b; padding: 14px 18px; border-radius: 10px;
                        border: 1px solid #475569; margin-top: 8px; max-width: 600px;'>
                <div style='color: {TEXT_COLOR}; font-size: 14px; font-weight: bold; margin-bottom: 8px;'>
                    Breakdown of top {top_n} features by role:
                </div>
                <div style='display: flex; gap: 20px;'>
                    <span style='color: {GREEN}; font-size: 14px;'>Causal: {n_causal}</span>
                    <span style='color: {RED}; font-size: 14px;'>Spurious: {n_spurious}</span>
                    <span style='color: {GRAY}; font-size: 14px;'>Incidental: {n_incidental}</span>
                </div>
                <div style='color: {MUTED_TEXT}; font-size: 12px; margin-top: 8px;'>
                    OLS assigns large coefficients to many spurious and incidental features —
                    it cannot distinguish meaningful predictors from noise.
                </div>
            </div>
            """)
            display(summary)

    def create_ui(self):
        title = widgets.HTML(f"""
        <h3 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>OLS Feature Importance</h3>
        <p style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 0;'>
            See which features OLS considers most important (by |coefficient|), color-coded by role.
            <b style='color: {GREEN};'>C</b>=Causal,
            <b style='color: {RED};'>S</b>=Spurious,
            <b style='color: {GRAY};'>I</b>=Incidental.
        </p>
        """)
        controls = widgets.HBox([self.n_slider, self.show_btn])
        return widgets.VBox([title, controls, self.output])

In [ ]:
#@title ▶️ Launch OLS Top Features { display-mode: "form" }

ols_feat_widget = OLSTopFeaturesWidget(X, y, codebook, role_map)
display(ols_feat_widget.create_ui())

---
## Takeaway

In [ ]:
#@title 🔧 Takeaway { display-mode: "form" }

display(takeaway_box(
    f"With {X.shape[1]} features and {X.shape[0]} countries, OLS memorizes training data perfectly "
    "(R²=1.0) but predicts terribly. <b>Memorization ≠ learning.</b> Can we fix this?"
))